# Encodage Catégoriel
Entrée : `metadata_features.parquet` → Sortie : `features_encodees.parquet`

- Suppression des colonnes string constantes, redondantes ou à trop haute cardinalité
- One-hot encoding des colonnes catégorielles utiles (drop_first=True)

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

ROOT           = Path().resolve().parent.parent
DATA_PROCESSED = ROOT / 'data' / 'processed'

df = pd.read_parquet(DATA_PROCESSED / 'metadata_features.parquet')
df_enc = df.copy()

print('Dataset chargé :', df_enc.shape)

## 1. Suppression des colonnes string inutiles

In [ ]:
DROP_REDUNDANT = [
    'in.building_america_climate_zone',
    'in.cec_climate_zone',
    'in.energystar_climate_zone_2023',
    'in.ashrae_iecc_climate_zone_2004_sub_cz_split',
    'in.hvac_heating_type_and_fuel',
    'in.census_division',
    'in.census_division_recs',
    'in.geometry_building_type_acs',
    'in.location_region',
    'in.ahs_region',
    'in.iso_rto_region',
    'in.puma_metro_status',
    'in.county_metro_status',
    'in.generation_and_emissions_assessment_region',
    'in.metropolitan_and_micropolitan_statistical_area',
    'in.reeds_balancing_area',
    'in.county_and_puma',
    'in.county_name',
    'in.county',
    'in.puma',
    'in.city',
    'in.state',
    'in.weather_file_city',
    'in.heating_unavailable_period',
    'in.cooling_unavailable_period',
    'in.upgrade_name',
]

str_cols = df_enc.select_dtypes(exclude=[np.number]).columns
drop_const    = [c for c in str_cols if df_enc[c].nunique() <= 1]
drop_explicit = [c for c in DROP_REDUNDANT if c in df_enc.columns]

to_drop = list(set(drop_const + drop_explicit))
df_enc.drop(columns=to_drop, inplace=True)

print(f'Colonnes constantes supprimées  : {len(drop_const)}')
print(f'Colonnes redondantes supprimées : {len(drop_explicit)}')
print(f'Shape : {df_enc.shape}')

## 2. Imputation par groupe (zone climatique)
Doit s'exécuter ici — avant le OHE — pendant que la zone climatique est encore une chaîne.

In [ ]:
GROUPBY_COL = 'in.ashrae_iecc_climate_zone_2004'

num_cols = df_enc.select_dtypes(include=[np.number]).columns
nan_cols = [c for c in num_cols if df_enc[c].isna().any()]

if nan_cols and GROUPBY_COL in df_enc.columns:
    for c in nan_cols:
        global_med  = df_enc[c].median()
        group_meds  = df_enc.groupby(GROUPBY_COL)[c].transform('median')
        df_enc[c]   = df_enc[c].fillna(group_meds.fillna(global_med))
    print(f'Imputation par zone climatique : {len(nan_cols)} colonnes')
    print(f'NaN résiduels : {df_enc[nan_cols].isna().sum().sum()}')
else:
    print('Colonne de groupement absente — ignoré.')

## 3. One-hot encoding des colonnes catégorielles restantes

In [ ]:
str_cols = df_enc.select_dtypes(exclude=[np.number]).columns.tolist()
str_cols = [c for c in str_cols if not c.startswith('out.')]

print(f'Colonnes string à encoder : {len(str_cols)}')
for c in sorted(str_cols):
    print(f'  {c:<60} nunique={df_enc[c].nunique()}')

for c in str_cols:
    df_enc[c] = df_enc[c].astype('category')

df_enc = pd.get_dummies(df_enc, columns=str_cols, dtype=np.int8, drop_first=True)

print(f'\nShape après one-hot : {df_enc.shape}')

## 4. Sauvegarde → `features_encodees.parquet`

In [ ]:
out_path = DATA_PROCESSED / 'features_encodees.parquet'
df_enc.to_parquet(out_path, index=False)

print(f'Sauvegardé : {out_path}')
print(f'Shape final : {df_enc.shape}')